In [17]:
'''
Bibliotecas usadas
'''

import os
import re
import subprocess
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from Bio import Entrez
from Bio import SeqIO
from Bio.Seq import Seq
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import subprocess
import shap

warnings.filterwarnings("ignore")

print("Bibliotecas importadas")


Bibliotecas importadas


In [18]:
'''
obtenção dos datasets
'''

EMAIL = "marcela.leite@gmail.com.br"
OUTPUT_DIR = "pipelineS"
os.makedirs(OUTPUT_DIR, exist_ok=True)
Entrez.email = EMAIL

# ==========================================================
# QUERIES
# ==========================================================

POSITIVE_QUERY = r'''
("Solanum lycopersicum"[Organism] OR "Nicotiana benthamiana"[Organism] OR "Capsicum annuum"[Organism] OR "Arabidopsis thaliana"[Organism])
AND( TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV  OR TCSV  OR CMV OR ToCV OR TICV OR AMV
OR "virus resistance" OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2" OR "RCY1"
OR "Rx" OR "R protein"  OR "NLR" OR "NBS-LRR"  OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "RLK" OR "RLP" OR "LRR receptor kinase"
OR "leucine rich repeat receptor kinase" OR "plant disease resistance protein" OR "disease resistance protein" OR "RPM1" OR "RPS2" OR "RPS4" OR "RPS5"
OR "RPP" OR "SNC1"  OR "ADR1" OR "NDR1" OR "EDS1" OR "PAD4" )
NOT ( partial OR fragment OR hypothetical OR predicted OR DAP-seq OR "transcription factor" OR DNA-binding )
'''
#OR microtom 
NEGATIVE_QUERY = r'''
( "Solanum lycopersicum"[Organism] OR "Nicotiana benthamiana"[Organism] OR "Capsicum annuum"[Organism] OR "Arabidopsis thaliana"[Organism])
AND ( "actin" OR "tubulin" OR "ribosomal protein" OR "ATP synthase" OR "photosystem" OR "elongation factor" OR "glycolysis" OR "metabolic enzyme" )
NOT ( "virus resistance"  OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2"
OR "RCY1" OR "Rx" OR "R protein" OR "resistance" OR "NLR" OR "LRR" OR "immune" OR "virus" )
'''

# Dados dos transcriptomas
# incluí nos dados de treino
# ARABIDOPSIS_QUERY = '''
#   "Arabidopsis thaliana"[Organism] AND
#       (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
#       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
#       AND 1500:30000[Sequence Length]
#       '''

SOLANUM_BETACEUM = '''
    Solanum betaceum [Organism]
AND (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
       AND 1500:30000[Sequence Length]
    '''

SOLANUM_MELONGENA = '''
    Solanum melongena [Organism]
AND (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
       AND 1500:30000[Sequence Length]
    '''

TUBEROSUM_QUERY = '''
    Solanum tuberosum [Organism]
AND (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
       AND 1500:30000[Sequence Length]
    '''

PHYSALIS_QUERY = '''
  "Physalis peruviana"[Organism] AND
      (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
      NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment OR microtom)
      AND 1500:30000[Sequence Length]
      '''


# ==========================================================
# DOWNLOAD NCBI
# ==========================================================

def baixar_proteinas_ncbi(query, output_fasta, max_records=5000):
    print("\n================================")
    print("DOWNLOAD NCBI")
    print("================================")
    handle = Entrez.esearch(
        db="protein",
        term=query,
        retmax=max_records
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Proteínas encontradas: {len(ids)}")
    if len(ids) == 0:
        return
    handle = Entrez.efetch(
        db="protein",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")

def baixar_transcriptoma(output_fasta, query):
    print("\n================================")
    print("DOWNLOAD TRANSCRIPTOMA")
    print("================================")
    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=10000
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Transcritos encontrados: {len(ids)}")
    handle = Entrez.efetch(
        db="nucleotide",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")


#Dados de Entrada - baixar arquivos FASTA das sequências de proteínas
POSITIVE_FASTA = f"{OUTPUT_DIR}/positive.fasta"
NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative.fasta"

baixar_proteinas_ncbi(POSITIVE_QUERY,POSITIVE_FASTA)
baixar_proteinas_ncbi(NEGATIVE_QUERY,NEGATIVE_FASTA)


print("Arquivos salvos na pasta: ",OUTPUT_DIR)
print("============ FIM COLETA DE DADOS TREINO/VALIDAÇÂO ============")



DOWNLOAD NCBI
Proteínas encontradas: 5000
Arquivo salvo: pipelineR/positive.fasta

DOWNLOAD NCBI
Proteínas encontradas: 5000
Arquivo salvo: pipelineR/negative.fasta
Arquivos salvos na pasta:  pipelineR
============ FIM COLETA DE DADOS TREINO/VALIDAÇÂO ============


In [19]:
#Dados para aplicação - baixar transcriptomas - RNA-Seq
# transcriptoma_arabidopsis_fasta = (f"{OUTPUT_DIR}/arabidopsis_transcriptoma.fasta")
transcriptoma_betaceum_fasta = (f"{OUTPUT_DIR}/betaceum_transcriptoma.fasta")
transcriptoma_melongena_fasta = (f"{OUTPUT_DIR}/melongena_transcriptoma.fasta")
transcriptoma_tuberosum_fasta = (f"{OUTPUT_DIR}/tuberosum_transcriptoma.fasta")
transcriptoma_physalis_fasta = (f"{OUTPUT_DIR}/physalis_transcriptoma.fasta")

# baixar_transcriptoma(transcriptoma_arabidopsis_fasta, ARABIDOPSIS_QUERY)
baixar_transcriptoma(transcriptoma_betaceum_fasta, SOLANUM_BETACEUM)
baixar_transcriptoma(transcriptoma_melongena_fasta, SOLANUM_MELONGENA)
baixar_transcriptoma(transcriptoma_tuberosum_fasta, TUBEROSUM_QUERY)
baixar_transcriptoma(transcriptoma_physalis_fasta, PHYSALIS_QUERY)
print("Arquivos salvos na pasta: ",OUTPUT_DIR)
print("============ FIM COLETA DE DADOS APLICAÇÃO ============")


DOWNLOAD TRANSCRIPTOMA
Transcritos encontrados: 0
Arquivo salvo: pipelineR/betaceum_transcriptoma.fasta

DOWNLOAD TRANSCRIPTOMA
Transcritos encontrados: 8589
Arquivo salvo: pipelineR/melongena_transcriptoma.fasta

DOWNLOAD TRANSCRIPTOMA
Transcritos encontrados: 300
Arquivo salvo: pipelineR/tuberosum_transcriptoma.fasta

DOWNLOAD TRANSCRIPTOMA
Transcritos encontrados: 10000
Arquivo salvo: pipelineR/physalis_transcriptoma.fasta
Arquivos salvos na pasta:  pipelineR
============ FIM COLETA DE DADOS APLICAÇÃO ============


In [20]:
'''
Tratamento dos dados
    - Criar Dataframes com os conjuntos de dados de treino e validação
    - Executar CD-Hit para eliminar sequencias muitos semelhantes
    - Extração de atributos (features)

'''

# ==========================================================
# AAC
# ==========================================================
# composição de aminoácidos - transformar a proteína em dados numéricos
# conta a frequência de cada aminoácido em uma sequẽncia de proteína
# como são 20 aminoácidos possíveis, irá gerar 20 features com seus percentuais
# com essa frequência consegue ter um parâmetro para distinguir as proteínas
# por exemplo proteínas de resistência tem geralmente certa frequência de aminoácidos
# mas ignora a ordem - por isso a sequẽncia de dipeptídeos irá incrementar o modelo considerando a ordem
# sequência de 20 aminoácidos
AMINO_ACIDOS = list("ACDEFGHIKLMNPQRSTVWY")

motifs = {
    # "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
    # "GLPL": r"GLPL",
    # "MHD": r"MHD"

    # # novos motifs
    # "KINASE2": r"[LIVM]{3,4}DD[VLIM]",
    # "RNBS_A": r"FDLxKxWVSVSD",
    # "RNBS_B": r"[A-Z]{2}G[PH][A-Z]{2}",
    # "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
    # "RNBS_D": r"CFLYCALFPED",
    # "VVVLDDVW": r"VVVLDDVW"
    # "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
    "P_LOOP": r"G[PAST][A-Z]{4}GK[ST]",
    # "KINASE2": r"[LIVM]{3,5}DD",
    "KINASE2": r"[LIVM]{4}DD[VIL]",
    "RNBS_A": r"FDL",
    "RNBS_B": r"G[RK]G",
    "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
    "GLPL": r"GLPL",
    "RNBS_D": r"CFL",
    "MHD": r"MHD",
    "VVVLDDVW": r"VVVLDDVW"
}

def calcular_aac(seq):
    seq = seq.upper()
    total = len(seq)
    if total == 0:
        return [0] * len(AMINO_ACIDOS)
    counts = Counter(seq)
    return [
        counts.get(aa, 0) / total
        for aa in AMINO_ACIDOS
    ]

# ==========================================================
# DIPEPTÍDEOS
# ==========================================================
'''
encontrar pares de aminoácidos consectíveis em uma sequência protéica  -sequência de dois aminoácidos
considera a ordem dos aminoácidos
há certos padrões comuns para 
para 20 aminoácidos, há possibilidade de 20*20 = 400 dipeptídeos - features
'''

DIPEPTIDES = [
    a + b
    for a in AMINO_ACIDOS
    for b in AMINO_ACIDOS
]

def calcular_dipeptideos(seq):
    seq = seq.upper()
    total = len(seq) - 1
    if total <= 0:
        return [0] * len(DIPEPTIDES)
    counts = Counter([
        seq[i:i+2]
        for i in range(total)
    ])
    return [
        counts.get(dp, 0) / total
        for dp in DIPEPTIDES
    ]

def localizar_motivos(seq):
    posicoes = {}
    for nome, pattern in motifs.items():
        match = re.search(pattern, seq)

        if match:
            posicoes[nome] = match.start()
        else:
            posicoes[nome] = None

    return posicoes
    

def calcular_distancias(posicoes):

    def distancia(a, b):
        if posicoes[a] is None or posicoes[b] is None:
            return -1
        return posicoes[b] - posicoes[a]

    return {
        "dist_ploop_kinase2": distancia("P_LOOP", "KINASE2"),
        "dist_kinase2_glpl": distancia("KINASE2", "GLPL"),
        "dist_glpl_mhd": distancia("GLPL", "MHD")
    }
    
def contar_motivos(posicoes):
    return sum(
        1 for v in posicoes.values()
        if v is not None
    )
    
def extrair_motifs(seq):

    return [
        int(bool(re.search(pattern, seq)))
        for pattern in motifs.values()
    ]



# ==========================================================
# FEATURES - atributos
# ==========================================================
#calculando features
def extrair_features_proteina(seq):
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    if len(seq) < 50:
        return None
    features = []
    features.append(len(seq)) # 1 - comprimento da cadeia
    features.extend(calcular_aac(seq)) # 20 composição de aminoácidos - frequência de cada um na sequência
    features.extend(calcular_dipeptideos(seq)) #400 possibilidades - considera a ordem dos aminoácidos para buscar os mais frequêntes em proteínas de resistência
    
    features.extend(extrair_motifs(seq))
    # posições dos motivos
    posicoes = localizar_motivos(seq)
    # número de motivos encontrados
    num_motifs_detectados = contar_motivos(posicoes)
    # distâncias entre motivos
    distancias = calcular_distancias(posicoes)
    
    features.append(num_motifs_detectados)
    protein_length = len(seq)

    features.append(distancias["dist_ploop_kinase2"])
    features.append(distancias["dist_kinase2_glpl"])
    features.append(distancias["dist_glpl_mhd"])
    
    features.append(
        distancias["dist_ploop_kinase2"]/protein_length
        if distancias["dist_ploop_kinase2"] != -1 else -1
    )
    features.append(
        distancias["dist_kinase2_glpl"]/protein_length
        if distancias["dist_kinase2_glpl"] != -1 else -1
    )
    features.append(
        distancias["dist_glpl_mhd"]/protein_length
        if distancias["dist_glpl_mhd"] != -1 else -1
    )
        
    features.append(int(posicoes["P_LOOP"] is not None))
    features.append(int(posicoes["KINASE2"] is not None))
    features.append(int(posicoes["GLPL"] is not None))
    features.append(int(posicoes["MHD"] is not None))
    
    return features
    
# ==========================================================
# DATASET
# ==========================================================
motif_cols = [
    "P_LOOP",
    "KINASE2",
    "RNBS_A",
    "RNBS_B",
    "RNBS_C",
    "GLPL",
    "RNBS_D",
    "MHD",
    "VVVLDDVW"
]

def criar_dataset(positive_fasta, negative_fasta):
    data = []
    columns = (
        ["protein_length"]
        + [f"AAC_{aa}" for aa in AMINO_ACIDOS]
        + [f"DIPEP_{dp}" for dp in DIPEPTIDES]
        + motif_cols
        + ["num_motifs_detectados"]
        + ["dist_ploop_kinase2"]
        + ["dist_kinase2_glpl"]
        + ["dist_glpl_mhd"]
        + ["dist_ploop_kinase2_norm"]
        + ["dist_kinase2_glpl_norm"]
        + ["dist_glpl_mhd_norm"]        
        + ["has_ploop"]
        + ["has_kinase2"]
        + ["has_glpl"]
        + ["has_mhd"]
    )

    # POSITIVOS
    for record in SeqIO.parse(positive_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [1, record.id]
        data.append(row)

    # NEGATIVOS
    for record in SeqIO.parse(negative_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [0, record.id]
        data.append(row)
        
    df = pd.DataFrame(
        data,
        columns=columns + ["target", "id"]
    )
    print("\nTotal de Features/Atributos:",len(columns)) # quantidade
    print("\nFeatures/atributos:")
    print(columns)     
    return df, columns

# CD-Hit
# chama o programa CD-HIT do sistema operacional
def executar_cdhit(
    input_fasta,
    output_fasta,
    identity=0.7  #0.5
):
    cmd = [
        "cd-hit",
        "-i", input_fasta,
        "-o", output_fasta,
        "-c", str(identity),
        "-n", "5",
        "-M", "30000",
        "-T", "4"
    ]

    print("\nExecutando CD-HIT...")
    subprocess.run(cmd, check=True)
    print("CD-HIT concluído.")

def limpar(seq_id):
    seq_id = seq_id.strip()
    seq_id = seq_id.split()[0]
    return seq_id

def ler_clusters_cdhit(cluster_file):
    clusters = {}
    cluster_id = None
    with open(cluster_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">Cluster"):
                cluster_id = line.replace(">Cluster ", "")
                clusters[cluster_id] = []
            else:
                seq_id = line.split(">")[1].split("...")[0]
                seq_id = limpar(seq_id)
                clusters[cluster_id].append(seq_id)
    return clusters

# executa o CD-Hit nos arquivos baixados
executar_cdhit(
    POSITIVE_FASTA,
    f"{OUTPUT_DIR}/positive_nr.fasta",
    identity=0.7
)

executar_cdhit(
    NEGATIVE_FASTA,
    f"{OUTPUT_DIR}/negative_nr.fasta",
    identity=0.7
)
# ======================================================
# Montar os DATASETs
# ======================================================

POSITIVE_FASTA = f"{OUTPUT_DIR}/positive_nr.fasta"
NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative_nr.fasta"


df, columns = criar_dataset(
    POSITIVE_FASTA,
    NEGATIVE_FASTA
)

df["id"] = df["id"].apply(limpar)
# ======================================================
# CLUSTERS
# ======================================================

clusters_pos = ler_clusters_cdhit(
    f"{OUTPUT_DIR}/positive_nr.fasta.clstr"
)

clusters_neg = ler_clusters_cdhit(
    f"{OUTPUT_DIR}/negative_nr.fasta.clstr"
)

cluster_map = {}

for c, seqs in clusters_pos.items():
    for s in seqs:
        cluster_map[s] = f"POS_{c}"

for c, seqs in clusters_neg.items():
    for s in seqs:
        cluster_map[s] = f"NEG_{c}"

df["cluster"] = df["id"].map(cluster_map)

print("\n Clusters ausentes: ")
print(df["cluster"].isna().sum())
df = df.dropna(subset=["cluster"])

print("\nDataset:", df.shape)

print("Fim tratamento dos dados...")




Executando CD-HIT...
Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i pipelineR/positive.fasta -o
         pipelineR/positive_nr.fasta -c 0.7 -n 5 -M 30000 -T 4

Started: Mon Jun  8 15:24:52 2026
                            Output                              
----------------------------------------------------------------
total seq: 5000
longest and shortest : 2604 and 12
Total letters: 3650305
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 4M
Buffer          : 4 X 11M = 44M
Table           : 2 X 65M = 130M
Miscellaneous   : 0M
Total           : 179M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 3727547274

# comparing sequences from          0  to        833
---------- new table with      224 representatives
# comparing sequences from        833  to       1527
----------    257 remaining sequences to the next cycle
---------- new table with     

TypeError: list indices must be integers or slices, not str

In [7]:
'''
k-Fold-Cross-Validation

'''

def validacao_cruzada(nome, model, X_train, y_train, groups_train):
    
    cv = StratifiedGroupKFold(
        n_splits=5, #10
        shuffle=True,
        random_state=42
    )
    
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv,
        scoring=[
            'accuracy',
            'precision',
            'recall',
            'f1',
            'roc_auc'
        ],
        n_jobs=-1
    )
    return scores  
  

In [15]:
'''
Criação dos modelos/ Treinamento
'''
# para melhor separar os conjuntos de treino e teste considerando os agrupamentos feitos no CD-HIT para sequẽncias muito parecidas
def split_por_homologia(df):
    print(df.head(30))
    groups = df["cluster"]
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )
    # cv  = StratifiedGroupKFold(
    #     n_splits = 10,
    #     shuffle=True,
    #     random_state = 42
    # )    

    train_idx, test_idx = next(
        splitter.split(df, groups=groups)
        # cv.split(df, df["target"], groups)
    )
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    return train_df, test_df

def calcular_e_salvar_shap(nome, model, X_test, feature_names, output_dir):
    """
    Calcula os SHAP values para modelos de árvore e salva os gráficos explicativos.
    """
    print(f"\n[SHAP] Inicializando análise de interpretabilidade para: {nome}...")
    
    # 1. Inicializa o explicador otimizado para árvores
    explainer = shap.TreeExplainer(model)
    
    # 2. Calcula os SHAP values para o conjunto de teste
    shap_values = explainer.shap_values(X_test)
    
    # 3. Tratamento de formato (Scikit-Learn Random Forest vs XGBoost)
    # O Random Forest do sklearn retorna uma lista contendo [valores_classe_0, valores_classe_1]
    # O XGBoost retorna diretamente a matriz de SHAP values para a classe alvo
    if isinstance(shap_values, list):
        # Selecionamos o índice 1 (classe positiva: proteínas de resistência/alvos)
        shap_values_pos = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
        # Tratamento para versões específicas do SHAP que geram arrays 3D
        shap_values_pos = shap_values[:, :, 1]
    else:
        shap_values_pos = shap_values

    # 4. Gráfico 1: Summary Plot (Beeswarm)
    # Mostra o impacto biológico de cada feature (ex: se alta frequência aumenta ou diminui a predição)
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_pos, 
        X_test, 
        feature_names=feature_names, 
        max_display=20,  # Foco nas top 20 características
        show=False
    )
    # plt.title(f"SHAP Summary Plot (Top 20) - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_summary.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    # 5. Gráfico 2: Bar Plot (Importância Global)
    # Mostra a magnitude média do impacto de cada feature de forma absoluta
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_pos, 
        X_test, 
        feature_names=feature_names, 
        plot_type="bar", 
        max_display=20, 
        show=False
    )
    plt.title(f"SHAP Global Feature Importance - {nome}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{nome}_shap_bar.png", bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"[SHAP] Gráficos salvos com sucesso em '{output_dir}/'!")

# ==========================================================
# TREINAMENTO - treina os modelos
# ==========================================================

def avaliar_modelo(nome, model, X_train, y_train, X_test, y_test):
    print("\n============================================================")
    print(nome)
    print("============================================================")
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= 0.5).astype(int)
    print(classification_report(y_test, preds))
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    mcc = matthews_corrcoef(y_test, preds)
    f1 = f1_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    bal_acc = balanced_accuracy_score(y_test, preds)

    print("ROC-AUC          :", roc)
    print("PR-AUC           :", pr)
    print("MCC              :", mcc)
    print("F1               :", f1)
    print("Precision        :", precision)
    print("Recall           :", recall)
    print("Balanced Accuracy:", bal_acc)

    # ======================================================
    # MATRIZ DE CONFUSÃO
    # ======================================================
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5,5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negativo", "Positivo"]
    )
    disp.plot(cmap="Blues")
    # plt.title(f"Matriz de Confusão - {nome}")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_matriz_confusao.png",
        bbox_inches="tight"
    )
    plt.close()
    # ======================================================
    # CURVA ROC
    # ======================================================
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,6))
    plt.plot(
        fpr,
        tpr,
        label=f"AUC = {roc_auc:.4f}"
    )
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    # plt.title(f"Curva ROC - {nome}")
    plt.legend(loc="lower right")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_roc.png",
        bbox_inches="tight"
    )
    plt.close()
    
    return {
        "Modelo": nome,
        "ROC_AUC": roc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Balanced_Accuracy": bal_acc
    }, model


# ======================================================
# Separação Treino/Teste 80/20
# SPLIT POR HOMOLOGIA
# ======================================================

train_df, test_df = split_por_homologia(df)
X_train = train_df[columns]
y_train = train_df["target"]
X_test = test_df[columns]
y_test = test_df["target"]
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ======================================================
# MODELOS
# ======================================================

rf = RandomForestClassifier(
    n_estimators=1000,
    max_depth = 12,
    min_samples_split=4,
    max_features='sqrt',
    random_state=42,
    class_weight="balanced",
    n_jobs = -1
)    
xgb = XGBClassifier(
    n_estimators=1200,
    max_depth=5,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    min_child_weight=3,
    reg_alpha=0.5,
    reg_lambda=2,
    scale_pos_weight=1.2
)
svm = SVC(
    probability=True,
    kernel="rbf",
    class_weight="balanced",
    random_state=42,
    C=2,
    gamma='scale'
)
scores = []
groups_train = train_df["cluster"]

scores_rf = validacao_cruzada('RF',rf,X_train, y_train, groups_train)
scores_xgb = validacao_cruzada('XGB',xgb, X_train, y_train, groups_train)
scores_svm = validacao_cruzada('SVM',svm, X_train, y_train, groups_train)

print("Cross-Validation:RF")
for k,v in scores_rf.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")

print("Cross-Validation XGB")
for k,v in scores_xgb.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")

print("Cross-Validation SVM")
for k,v in scores_svm.items():
    if "test" in k:
        print(f"{k}: {v.mean():.4f} ± {v.std():.4f}")


resultados_cv = pd.DataFrame({
    'Modelo': ['RF','XGB','SVM'],
    'Accuracy': [
        scores_rf['test_accuracy'].mean(),
        scores_xgb['test_accuracy'].mean(),
        scores_svm['test_accuracy'].mean(),
    ],    
    'Recall': [
        scores_rf['test_recall'].mean(),
        scores_xgb['test_recall'].mean(),
        scores_svm['test_recall'].mean(),
    ],
    'Precision': [
        scores_rf['test_precision'].mean(),
        scores_xgb['test_precision'].mean(),
        scores_svm['test_precision'].mean(),
    ],
    'F1': [
        scores_rf['test_f1'].mean(),
        scores_xgb['test_f1'].mean(),
        scores_svm['test_f1'].mean(),
    ],
    'ROC_AUC': [
        scores_rf['test_roc_auc'].mean(),
        scores_xgb['test_roc_auc'].mean(),
        scores_svm['test_roc_auc'].mean(),
    ]
})
print(resultados_cv)

dados = []

for score in scores_rf['test_roc_auc']:
    dados.append(['RF',score])
for score in scores_xgb['test_roc_auc']:
    dados.append(['XGB',score])
for score in scores_svm['test_roc_auc']:
    dados.append(['SVM',score])

grafico = pd.DataFrame(dados, columns=['Modelo','ROC_AUC'])
grafico.boxplot(by='Modelo')
plt.ylabel("ROC-AUC")
# plt.title("10-Fold Cross Validation")
plt.suptitle("")
plt.tight_layout()
# plt.show()
plt.savefig(
    f"{OUTPUT_DIR}/kfold_cross_validation_roc.png",
    bbox_inches="tight"
)
plt.close()



################################################
# Treinamento

results = []
r1, rf_model = avaliar_modelo(
    "Random Forest",
    rf,
    X_train,
    y_train,
    X_test,
    y_test
) 
results.append(r1)
calcular_e_salvar_shap("Random Forest", rf_model, X_test, columns,OUTPUT_DIR )

r2, xgb_model = avaliar_modelo(
    "XGBoost",
    xgb,
    X_train,
    y_train,
    X_test,
    y_test
)
results.append(r2)
calcular_e_salvar_shap("XGBoost", xgb_model, X_test, columns,OUTPUT_DIR )

r3, svm_model = avaliar_modelo(
    "SVM",
    svm,
    X_train,
    y_train,
    X_test,
    y_test
)
results.append(r3)

print("\nFim criação/treino dos modelos\n")


     protein_length     AAC_A     AAC_C     AAC_D     AAC_E     AAC_F  \
0              1053  0.034188  0.018993  0.043685  0.038936  0.058879   
1               231  0.090909  0.012987  0.069264  0.077922  0.051948   
2               888  0.055180  0.018018  0.061937  0.072072  0.045045   
3               504  0.103175  0.019841  0.045635  0.071429  0.029762   
4               704  0.049716  0.007102  0.080966  0.117898  0.041193   
5               535  0.095327  0.009346  0.020561  0.026168  0.082243   
6               450  0.040000  0.028889  0.035556  0.082222  0.024444   
7               117  0.025641  0.017094  0.051282  0.068376  0.042735   
8                89  0.123596  0.078652  0.011236  0.112360  0.033708   
96              751  0.061252  0.014647  0.057257  0.055925  0.038615   
97              166  0.072289  0.024096  0.054217  0.072289  0.054217   
98              160  0.043750  0.062500  0.087500  0.062500  0.062500   
99              116  0.034483  0.086207  0.120690  

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

In [16]:
'''
Avaliação e Validação dos modelos
'''
# verificar importância dos atributos/feature para o Random Forest
importancia = pd.DataFrame({"feature":columns,"importance":rf_model.feature_importances_})
importancia = importancia.sort_values("importance",ascending = False)
print("\nTop 20 features: Random Forest")
print(importancia.head(20))
# GRÁFICO
top = importancia.head(20)
plt.figure(figsize=(10,8))
plt.barh( top["feature"][::-1], top["importance"][::-1] )
plt.xlabel("Importância")
plt.ylabel("Feature")
# plt.title("Top 20 Features - Random Forest")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/rf_importancia_variaveis.png",
    bbox_inches="tight"
)
plt.close()

# verificar importância dos atributos/feature para o XGBoost
importancia = pd.DataFrame({"feature":columns, "importance":xgb_model.feature_importances_})
importancia = importancia.sort_values("importance", ascending = False)
print("\nTop 20 features: XGBoost")
print(importancia.head(20))
# GRÁFICO
top = importancia.head(20)
plt.figure(figsize=(10,8))
plt.barh(top["feature"][::-1], top["importance"][::-1])
plt.xlabel("Importância")
plt.ylabel("Feature")
# plt.title("Top 20 Features - XGBoost")
plt.tight_layout()
plt.savefig(
    f"{OUTPUT_DIR}/xgb_importancia_variaveis.png",
    bbox_inches="tight"
)
plt.close()

print("\nCOMPARAÇÃO MODELOS")
print(pd.DataFrame(results))

results_df = pd.DataFrame(results)

metrics = [
    "ROC_AUC",
    "F1",
    "Precision",
    "Recall",
    "Balanced_Accuracy"
]

for metric in metrics:
    plt.figure(figsize=(8,5))
    plt.bar(
        results_df["Modelo"],
        results_df[metric]
    )

    plt.ylim(0, 1)
    plt.ylabel(metric)
    # plt.title(f"Comparação dos Modelos - {metric}")
    for i, v in enumerate(results_df[metric]):
        plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

    plt.savefig(
        f"{OUTPUT_DIR}/comparacao_{metric}.png",
        bbox_inches="tight"
    )
    plt.close()

modelos = {
    "Random Forest":rf_model,
    "XGBoost":xgb_model,
    "SVM":svm_model
    }
plt.figure(figsize=(7,7))
for nome, model in modelos.items():
    probs = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(
        fpr,
        tpr,
        label=f"{nome} AUC={roc_auc:.3f}"
    )
plt.plot([0,1],[0,1],'--')
plt.xlabel("FPR")
plt.ylabel("TPR")
# plt.title("Comparação ROC")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/roc_comparativa.png")
plt.close()

print("Fim avaliação / validação modelos ")


Top 20 features: Random Forest
            feature  importance
0    protein_length    0.032332
1             AAC_A    0.031940
10            AAC_L    0.020408
207        DIPEP_LH    0.018931
250        DIPEP_NL    0.017609
422         KINASE2    0.013113
65         DIPEP_DF    0.012780
106        DIPEP_FG    0.012756
50         DIPEP_CL    0.012366
421          P_LOOP    0.011541
100        DIPEP_EY    0.011392
410        DIPEP_YL    0.011127
272        DIPEP_PN    0.010040
248        DIPEP_NI    0.009991
21         DIPEP_AA    0.009592
85         DIPEP_EF    0.009402
210        DIPEP_LL    0.009171
167        DIPEP_IH    0.008588
45         DIPEP_CF    0.008304
380        DIPEP_VY    0.008141

Top 20 features: XGBoost
            feature  importance
0    protein_length    0.060213
422         KINASE2    0.016675
207        DIPEP_LH    0.011878
1             AAC_A    0.011428
250        DIPEP_NL    0.008655
10            AAC_L    0.007536
155        DIPEP_HR    0.007191
421          P

In [11]:
'''
Funções para rodar a aplicação
'''
# MOTIFS = {
#     # "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
#     # "GLPL": r"GLPL",
#     # "MHD": r"MHD"

#     # # novos motifs
#     # "KINASE2": r"[LIVM]{3,4}DD[VLIM]",
#     # "RNBS_A": r"FDLxKxWVSVSD",
#     # "RNBS_B": r"[A-Z]{2}G[PH][A-Z]{2}",
#     # "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
#     # "RNBS_D": r"CFLYCALFPED",
#     # "VVVLDDVW": r"VVVLDDVW"
#     "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
#     "KINASE2": r"[LIVM]{3,5}DD",
#     "RNBS_A": r"FDL",
#     "RNBS_B": r"G[RK]G",
#     "RNBS_C": r"[A-Z]{2}P[A-Z]{2}W",
#     "GLPL": r"GLPL",
#     "RNBS_D": r"CFL",
#     "MHD": r"MHD",
#     "VVVLDDVW": r"VVVLDDVW"
# }
    
def validar_motifs(df):
    print("\nVALIDAÇÃO BIOLÓGICA\n")
    counts = {}
    for motif_name, pattern in motifs.items():
        counts[motif_name] = 0
        for seq in df["protein_seq"]:
            if re.search(pattern, seq):
                counts[motif_name] += 1
    for k, v in counts.items():
        print(k, ":", v)


def rodar_transdecoder(fasta_nt):
    cmd1 = [
        "TransDecoder.LongOrfs",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output1.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd1, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)

    cmd2 = [
        "TransDecoder.Predict",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output2.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd2, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)
    pep_file = fasta_nt + ".transdecoder.pep"
    return pep_file


def predizer_transcritos(fasta_nt,model,nmodel, output):
    candidatos = []
    pep_file = rodar_transdecoder(fasta_nt)
    for record in SeqIO.parse(fasta_nt+".transdecoder.pep", "fasta"):
        protein = str(record.seq)
        feats = extrair_features_proteina(protein)
        if feats is None:
            continue
        X = scaler.transform([feats])
        prob = model.predict_proba(X)[0][1]
        candidatos.append({
            "id": record.id,
            "descricao": record.description,
            "protein_length": len(protein),
            "probabilidade_resistencia": prob,
            "protein_seq": protein
        })

    candidatos_df = pd.DataFrame(candidatos)
    candidatos_df = candidatos_df.sort_values(
        "probabilidade_resistencia",
        ascending=False
    )

    top = candidatos_df.head(20)
    print("\nTOP CANDIDATOS: "+fasta_nt+", MODELO: "+nmodel)
    print(top[[
        "id",
        "descricao",
        "protein_length",
        "probabilidade_resistencia"
    ]])

    validar_motifs(top)

    candidatos_df.to_csv(
        f"{OUTPUT_DIR}/{output}",
        index=False
    )


In [12]:
'''
Aplicação - Modelo XGBoost
'''

# print("\nPredição para ARABIDOPSIS\n")
# predizer_transcritos(transcriptoma_arabidopsis_fasta, xgb_model,'XGBoost',output="predicoes_xgboost.csv")

# print("\nPredição para BETACEUM\n")
# predizer_transcritos(transcriptoma_betaceum_fasta, xgb_model,'XGBoost',output="predicoes_xgboost.csv")

print("\nPredição para MELONGENA\n")
predizer_transcritos(transcriptoma_melongena_fasta, xgb_model,'XGBoost',output="predicoes_xgboost.csv")

print("\nPredição para TUBEROSUM\n")
predizer_transcritos(transcriptoma_tuberosum_fasta, xgb_model,'XGBoost',output="predicoes_xgboost.csv")

print("\nPredição para PHYSALIS\n")
predizer_transcritos(transcriptoma_physalis_fasta, xgb_model,'XGBoost',output="predicoes_xgboost.csv")

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_xgboost.csv")




Predição para MELONGENA


TOP CANDIDATOS: pipelineQ/melongena_transcriptoma.fasta, MODELO: XGBoost
                     id                                          descricao  \
3774  GBEF01024584.1.p1  GBEF01024584.1.p1 GENE.GBEF01024584.1~~GBEF010...   
40    GBEF01000145.1.p1  GBEF01000145.1.p1 GENE.GBEF01000145.1~~GBEF010...   
994   GBEF01014969.1.p1  GBEF01014969.1.p1 GENE.GBEF01014969.1~~GBEF010...   
8171  GBEF01041955.1.p1  GBEF01041955.1.p1 GENE.GBEF01041955.1~~GBEF010...   
5210  GBEF01027867.1.p1  GBEF01027867.1.p1 GENE.GBEF01027867.1~~GBEF010...   
286   GBEF01001123.1.p1  GBEF01001123.1.p1 GENE.GBEF01001123.1~~GBEF010...   
589   GBEF01002241.1.p1  GBEF01002241.1.p1 GENE.GBEF01002241.1~~GBEF010...   
36    GBEF01000129.1.p1  GBEF01000129.1.p1 GENE.GBEF01000129.1~~GBEF010...   
590   GBEF01002242.1.p1  GBEF01002242.1.p1 GENE.GBEF01002242.1~~GBEF010...   
6324  GBEF01032168.1.p1  GBEF01032168.1.p1 GENE.GBEF01032168.1~~GBEF010...   
9150  GBEF01044247.1.p1  GBEF01044247.1.p1

In [13]:
'''
Aplicação - Modelo Random Forest
'''
# print("\nPredição para ARABIDOPSIS\n")
# predizer_transcritos(transcriptoma_arabidopsis_fasta, rf_model,"RF",output="predicoes_rf.csv")

# print("\nPredição para BETACEUM\n")
# predizer_transcritos(transcriptoma_betaceum_fasta, rf_model,'RF',output="predicoes_rf.csv")

print("\nPredição para MELONGENA\n")
predizer_transcritos(transcriptoma_melongena_fasta, rf_model,'RF',output="predicoes_rf.csv")

print("\nPredição para TUBEROSUM\n")
predizer_transcritos(transcriptoma_tuberosum_fasta, rf_model,"RF",output="predicoes_rf.csv")

print("\nPredição para PHYSALIS\n")
predizer_transcritos(transcriptoma_physalis_fasta, rf_model,"RF",output="predicoes_rf.csv")

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_rf.csv")



Predição para MELONGENA


TOP CANDIDATOS: pipelineQ/melongena_transcriptoma.fasta, MODELO: RF
                     id                                          descricao  \
267   GBEF01001042.1.p1  GBEF01001042.1.p1 GENE.GBEF01001042.1~~GBEF010...   
1568  GBEF01016235.1.p1  GBEF01016235.1.p1 GENE.GBEF01016235.1~~GBEF010...   
589   GBEF01002241.1.p1  GBEF01002241.1.p1 GENE.GBEF01002241.1~~GBEF010...   
590   GBEF01002242.1.p1  GBEF01002242.1.p1 GENE.GBEF01002242.1~~GBEF010...   
994   GBEF01014969.1.p1  GBEF01014969.1.p1 GENE.GBEF01014969.1~~GBEF010...   
4989  GBEF01027318.1.p1  GBEF01027318.1.p1 GENE.GBEF01027318.1~~GBEF010...   
5625  GBEF01028897.1.p1  GBEF01028897.1.p1 GENE.GBEF01028897.1~~GBEF010...   
8171  GBEF01041955.1.p1  GBEF01041955.1.p1 GENE.GBEF01041955.1~~GBEF010...   
36    GBEF01000129.1.p1  GBEF01000129.1.p1 GENE.GBEF01000129.1~~GBEF010...   
9150  GBEF01044247.1.p1  GBEF01044247.1.p1 GENE.GBEF01044247.1~~GBEF010...   
3774  GBEF01024584.1.p1  GBEF01024584.1.p1 GENE

In [14]:
'''
Aplicação - Modelo SVM
'''

# print("\nPredição para ARABIDOPSIS\n")
# predizer_transcritos(transcriptoma_arabidopsis_fasta, svm_model,"SVM",output="predicoes_svm.csv")

# print("\nPredição para BETACEUM\n")
# predizer_transcritos(transcriptoma_betaceum_fasta, svm_model,'SVM',output="predicoes_svm.csv")

print("\nPredição para MELONGENA\n")
predizer_transcritos(transcriptoma_melongena_fasta, svm_model,'SVM',output="predicoes_svm.csv")

print("\nPredição para TUBEROSUM\n")
predizer_transcritos(transcriptoma_tuberosum_fasta, svm_model,"SVM",output="predicoes_svm.csv")

print("\nPredição para PHYSALIS\n")
predizer_transcritos(transcriptoma_physalis_fasta, svm_model,"SVM",output="predicoes_svm.csv")  

print("\nArquivo salvo:")
print(f"{OUTPUT_DIR}/predicoes_svm.csv")


Predição para MELONGENA


TOP CANDIDATOS: pipelineQ/melongena_transcriptoma.fasta, MODELO: SVM
                     id                                          descricao  \
590   GBEF01002242.1.p1  GBEF01002242.1.p1 GENE.GBEF01002242.1~~GBEF010...   
589   GBEF01002241.1.p1  GBEF01002241.1.p1 GENE.GBEF01002241.1~~GBEF010...   
6390  GBEF01032474.1.p1  GBEF01032474.1.p1 GENE.GBEF01032474.1~~GBEF010...   
286   GBEF01001123.1.p1  GBEF01001123.1.p1 GENE.GBEF01001123.1~~GBEF010...   
263   GBEF01001032.1.p1  GBEF01001032.1.p1 GENE.GBEF01001032.1~~GBEF010...   
8171  GBEF01041955.1.p1  GBEF01041955.1.p1 GENE.GBEF01041955.1~~GBEF010...   
4127  GBEF01025346.1.p1  GBEF01025346.1.p1 GENE.GBEF01025346.1~~GBEF010...   
994   GBEF01014969.1.p1  GBEF01014969.1.p1 GENE.GBEF01014969.1~~GBEF010...   
36    GBEF01000129.1.p1  GBEF01000129.1.p1 GENE.GBEF01000129.1~~GBEF010...   
967   GBEF01014919.1.p1  GBEF01014919.1.p1 GENE.GBEF01014919.1~~GBEF010...   
9150  GBEF01044247.1.p1  GBEF01044247.1.p1 GEN